## Standard Header

In [ ]:
# Cell 1 — Install dependencies
!pip install scipy numpy matplotlib torch torchvision \
    torch-geometric umap-learn wandb networkx tqdm -q

In [ ]:
# Cell 2 — Clone repo (re-clones every session; pulls latest if already exists)
import os
REPO_ROOT = '/content/antenna-gnn'
if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/asparagusD/antenna_gnn.git {REPO_ROOT}
else:
    !git -C {REPO_ROOT} pull --quiet
import sys
sys.path.insert(0, f'{REPO_ROOT}/src')   # makes 'from model import AntennaGNN' work
print(f'Repo ready at {REPO_ROOT}')

In [ ]:
# Cell 3 — Mount Drive and set data paths
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = '/content/drive/MyDrive/antenna_gnn'
RAW_DATA  = '/content/drive/MyDrive/antenna_dataset'
for d in [f'{DATA_ROOT}/artifacts', f'{DATA_ROOT}/checkpoints',
          f'{DATA_ROOT}/figures',   f'{DATA_ROOT}/splits',
          f'{DATA_ROOT}/data/processed']:
    os.makedirs(d, exist_ok=True)
print(f'Drive mounted. DATA_ROOT={DATA_ROOT}')

## Section 1: Load seed mask

In [ ]:
import numpy as np
import scipy.io as sio

seed_mask_25 = np.load(f'{DATA_ROOT}/artifacts/seed_mask_25.npy')
coords = np.argwhere(seed_mask_25)
seed_centroid_25 = tuple(coords.mean(axis=0))  # (row_c, col_c)
print(f"Seed centroid for 25x25: {seed_centroid_25}")

## Section 2: mat_to_networkx function

In [ ]:
import networkx as nx

def mat_to_networkx(patch_pattern, seed_mask):
    N = patch_pattern.shape[0]
    G = nx.DiGraph()
    
    # Compute seed centroid
    coords = np.argwhere(seed_mask)
    seed_r, seed_c = tuple(coords.mean(axis=0))
    
    # Add pixel nodes
    for i in range(N):
        for j in range(N):
            idx = i * N + j
            metal = int(patch_pattern[i, j])
            x_norm = j / (N - 1)
            y_norm = i / (N - 1)
            is_seed = int(seed_mask[i, j])
            dist_f = np.sqrt((i - seed_r)**2 + (j - seed_c)**2) / N
            G.add_node(idx, metal=metal, x_norm=x_norm, y_norm=y_norm,
                       is_seed=is_seed, dist_feed=dist_f, is_global=0)
            
    # Add virtual global node
    global_idx = 'global'
    G.add_node(global_idx, is_global=1, metal=0, x_norm=0.5, y_norm=0.5,
               is_seed=0, dist_feed=0.0)
               
    # Add 4-connectivity edges
    etype_map = {(1, 1): 0, (1, 0): 1, (0, 1): 2, (0, 0): 3}
    for i in range(N):
        for j in range(N):
            idx = i * N + j
            m_ij = int(patch_pattern[i, j])
            for (di, dj, direction) in [(0, 1, 0), (0, -1, 1), (-1, 0, 2), (1, 0, 3)]:
                ni, nj = i + di, j + dj
                if 0 <= ni < N and 0 <= nj < N:
                    nidx = ni * N + nj
                    m_nb = int(patch_pattern[ni, nj])
                    etype = etype_map[(m_ij, m_nb)]
                    G.add_edge(idx, nidx, etype=etype, direction=direction)
                    
    # Connect 'global' to all metal pixel nodes bidirectionally
    for i in range(N):
        for j in range(N):
            if patch_pattern[i, j] == 1:
                idx = i * N + j
                G.add_edge(global_idx, idx, etype=4, direction=4)
                G.add_edge(idx, global_idx, etype=4, direction=4)
                
    return G

## Section 3: draw_antenna_graph function

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def draw_antenna_graph(G, N, title, save_path):
    fig, ax = plt.subplots(figsize=(10, 10))
    
    pos = {}
    for node, data in G.nodes(data=True):
        if data.get('is_global', 0) == 1:
            pos[node] = (-0.15, 0.5)
        else:
            # Using (x_norm, y_norm) as requested
            pos[node] = (data['x_norm'], data['y_norm'])
            
    # Separate nodes
    metal_nodes = [n for n, d in G.nodes(data=True) if d.get('metal', 0) == 1 and d.get('is_global', 0) == 0]
    air_nodes = [n for n, d in G.nodes(data=True) if d.get('metal', 0) == 0 and d.get('is_global', 0) == 0]
    global_node = ['global']
    
    # Separate edges
    metal_edges = [(u, v) for u, v, d in G.edges(data=True) if d.get('etype') == 0]
    other_pixel_edges = [(u, v) for u, v, d in G.edges(data=True) if d.get('etype') in [1, 2, 3]]
    global_edges = [(u, v) for u, v, d in G.edges(data=True) if d.get('etype') == 4]
    
    # Draw edges
    nx.draw_networkx_edges(G, pos, edgelist=other_pixel_edges, edge_color='#EEEEEE', alpha=0.3, ax=ax)
    nx.draw_networkx_edges(G, pos, edgelist=metal_edges, edge_color='crimson', alpha=0.8, width=1.5, ax=ax)
    nx.draw_networkx_edges(G, pos, edgelist=global_edges, edge_color='royalblue', alpha=0.4, ax=ax)
    
    # Draw nodes
    nx.draw_networkx_nodes(G, pos, nodelist=metal_nodes, node_color='black', node_size=30, ax=ax, label='Metal pixel')
    nx.draw_networkx_nodes(G, pos, nodelist=air_nodes, node_color='#CCCCCC', node_size=10, ax=ax, label='Air pixel')
    nx.draw_networkx_nodes(G, pos, nodelist=global_node, node_color='royalblue', node_size=300, node_shape='*', ax=ax, label='Global node')
    
    # Legend and title
    ax.legend(scatterpoints=1, loc='upper right')
    ax.set_title(title)
    
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

## Section 4: Verification

In [ ]:
import glob

all_files_25 = sorted(glob.glob(
    f'{RAW_DATA}/training dataset/25x25/**/Mat_Files/*.mat',
    recursive=True))

file_func = None
file_nonfunc = None

for f in all_files_25:
    mat = sio.loadmat(f)
    is_func = mat['resonant_freqs'].size > 0
    if is_func and file_func is None:
        file_func = (f, mat)
    elif not is_func and file_nonfunc is None:
        file_nonfunc = (f, mat)
    if file_func is not None and file_nonfunc is not None:
        break

def verify_graph(mat, title, save_path):
    pattern = mat['patch_pattern']
    G = mat_to_networkx(pattern, seed_mask_25)
    draw_antenna_graph(G, 25, title, save_path)
    
    num_nodes = G.number_of_nodes()
    num_edges = G.number_of_edges()
    degree_global = G.degree('global')
    metal_count = int(np.sum(pattern))
    
    print(f"[{title}]")
    print(f"Nodes: {num_nodes}, Edges: {num_edges}, Global Degree: {degree_global}")
    
    assert num_nodes == 626
    assert degree_global == 2 * metal_count

verify_graph(file_func[1], "Functioning 25x25 Antenna", f'{DATA_ROOT}/figures/graph_vis_functioning.png')
verify_graph(file_nonfunc[1], "Non-functioning 25x25 Antenna", f'{DATA_ROOT}/figures/graph_vis_nonfunctioning.png')

## Section 5: Degree analysis

In [ ]:
import random

all_files_25 = sorted(glob.glob(
    f'{RAW_DATA}/training dataset/25x25/**/Mat_Files/*.mat',
    recursive=True))

random.seed(42)
sampled_paths = random.sample(all_files_25, 50)

global_degrees = []
metal_counts = []

for p in sampled_paths:
    mat = sio.loadmat(p)
    pattern = mat['patch_pattern']
    G = mat_to_networkx(pattern, seed_mask_25)
    
    degree = G.degree('global') // 2  # unidirectional equivalent
    metal_count = int(np.sum(pattern))
    
    global_degrees.append(degree)
    metal_counts.append(metal_count)
    
assert global_degrees == metal_counts, "Global degrees mismatch metal counts!"

combined_data = global_degrees + metal_counts
shared_bins = np.histogram_bin_edges(combined_data, bins=20)

plt.figure(figsize=(8, 6))
plt.hist(global_degrees, bins=shared_bins, histtype='step', linewidth=2.5, color='royalblue', label='Virtual Node Degree / 2')
plt.hist(global_degrees, bins=shared_bins, histtype='stepfilled', alpha=0.15, color='royalblue')
plt.hist(metal_counts, bins=shared_bins, histtype='step', linewidth=2.5, color='crimson', label='Metal Pixel Count')
plt.hist(metal_counts, bins=shared_bins, histtype='stepfilled', alpha=0.15, color='crimson')
plt.title('Virtual Node Degree vs Metal Pixel Count')
plt.xlabel('Count')
plt.ylabel('Frequency')
plt.legend()

plt.savefig(f'{DATA_ROOT}/figures/degree_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

max_diff = np.max(np.abs(np.array(global_degrees) - np.array(metal_counts)))
print(f"Max absolute difference: {max_diff}")


## Section 6: Scale test

In [ ]:
files_55 = sorted(glob.glob(
    f'{RAW_DATA}/fine-tuning dataset/55x55/**/Mat_Files/*.mat',
    recursive=True))

mat_55 = sio.loadmat(files_55[0])
pattern_55 = mat_55['patch_pattern']
seed_mask_55 = np.load(f'{DATA_ROOT}/artifacts/seed_mask_55.npy')

G_55 = mat_to_networkx(pattern_55, seed_mask_55)

num_nodes = G_55.number_of_nodes()
degree_global = G_55.degree('global')
metal_count = int(np.sum(pattern_55))
num_edges = G_55.number_of_edges()

print(f"55x55 Nodes: {num_nodes}")
print(f"55x55 Global Degree: {degree_global}")
print(f"55x55 Metal Count: {metal_count}")
print(f"55x55 Edges: {num_edges}")

assert num_nodes == 3026
assert degree_global == 2 * metal_count